# Figure 4

In [ ]:
library(data.table)
library(tidyverse)
library(colocboost)
gene_id = 'ENSG00000106460'
gene_name = 'TMEM106B'

## Reference annotation

In [ ]:
anno_ref <- fread( 'Annotation_with_gp_coordinates_clean.txt', sep = '\t')

## Data caches for the panels a (heatmap), b, c, d, e

### a heatmap data

In [ ]:
# Panel a (lower): detection of T185 / S185 forms of the N183 high-mannose glycopeptides
suppressMessages({library(SummarizedExperiment); library(data.table)})
glyco <- readRDS("tmem106b/glyco/glyco_tmem106b.Rds")
genotypes <- c("C/C", "C/G", "G/G")
keep <- colData(glyco)$rs3173615 %in% genotypes & !is.na(colData(glyco)$tangles_sqrt)
glyco <- glyco[, keep]; gt <- factor(colData(glyco)$rs3173615, levels = genotypes); stopifnot(ncol(glyco) == 316)
pairs <- list(Hex5 = c(T = "TMEM_GP04", S = "TMEM_GP02"), Hex6 = c(T = "TMEM_GP10", S = "TMEM_GP08"),
              Hex7 = c(T = "TMEM_GP16", S = "TMEM_GP14"), Hex8 = c(T = "TMEM_GP19", S = "TMEM_GP18"))
I <- assay(glyco, "intensity"); M <- assay(glyco, "match_type"); L <- assay(glyco, "localization")
localized <- matrix(grepl("N:[0-9.]+@LNN\\(1\\)", L), nrow = nrow(glyco))
direct <- I > 0 & M == "MS/MS" & assay(glyco, "spectral_count") > 0 & localized
detected <- I > 0 & (direct | M == "MBR"); detected[is.na(detected)] <- FALSE
counts <- rbindlist(lapply(names(pairs), function(h) {
  Tt <- detected[pairs[[h]]["T"], ]; Ss <- detected[pairs[[h]]["S"], ]
  rbindlist(lapply(genotypes, function(g) { ix <- gt == g
    data.table(glycan = h, genotype = g, signal = c("T185", "S185", "Any"), n = c(sum(Tt[ix]), sum(Ss[ix]), sum((Tt | Ss)[ix])), N = sum(ix)) }))
}))
counts[, fraction := n / N] %>% head
saveRDS(counts, "data/Figure_4a_heatmap.rds")

### b peptide pQTL effects (`data/Figure_4b_pqtl_effects.rds`)

In [ ]:
# Panel b effects: same data preparation and model as figureS4c.R
tmt <- readRDS('tmem106b/tmt/tmt_tmem106b.Rds')
d <- as.data.frame(SummarizedExperiment::colData(tmt))
ix <- order(match(d$Round, c('r01', 'r03', 'r02')), d$Batch, seq_len(nrow(d))); ix <- ix[!duplicated(d$ProjID[ix])]
donors <- d[ix, ] %>% mutate(Batch = factor(Batch), study = factor(study), sex = factor(msex), age10 = (age_death - 90) / 10,
                             genotype = factor(rs3173615, levels = c('C/C', 'C/G', 'G/G')), dosage = as.numeric(genotype) - 1)
abund <- SummarizedExperiment::assay(tmt, 'peptides')[, ix]
peptides <- c('EDAYDGVTSENMR', 'NGDVSQFPYVEFTGR', 'DSVTCPTCQGTGR', 'GQENQLVALIPYSDQR', 'SAYVSYDVQK', 'YQYVDCGR')
fit_peptide <- function(p) {
  y <- abund[p, ]; z <- (y - mean(y, na.rm = TRUE)) / sigma(lm(y ~ Batch, data = donors))
  v <- donors %>% mutate(z = z) %>% tidyr::drop_na(z, genotype, Batch, study, age10, sex)
  fit <- lm(z ~ genotype + Batch + study + age10 + sex, data = v)
  joint <- car::linearHypothesis(fit, c('genotypeC/G = 0', 'genotypeG/G = 0'), vcov. = sandwich::vcovHC(fit, type = 'HC3'), test = 'F')
  fit_add <- lm(z ~ dosage + Batch + study + age10 + sex, data = v)
  tibble(peptide = p, n = nrow(v), p_joint = joint[2, 'Pr(>F)'], beta = coef(fit_add)[['dosage']], se = sqrt(sandwich::vcovHC(fit_add, type = 'HC3')['dosage', 'dosage']))
}
fits <- map_dfr(peptides, fit_peptide)
s4c <- readRDS('../../3_supplementary_figures/Supplementary_Figure4/data/Figure_S4c.rds')$results
stopifnot(isTRUE(all.equal(fits$p_joint[fits$peptide == 'SAYVSYDVQK'], s4c['Core', 'p_genotype_HC3'], check.attributes = FALSE)),
          isTRUE(all.equal(fits$p_joint[fits$peptide == 'YQYVDCGR'], s4c['Boundary', 'p_genotype_HC3'], check.attributes = FALSE)))
saveRDS(fits, 'data/Figure_4b_pqtl_effects.rds'); fits


### c colocboost data (locus zoom)

In [ ]:
# Panel c: locus zoom of the rerun
suppressMessages(library(colocboost))
win <- c(12171997, 12292440)
cb <- readRDS("data_output/TMEM106B_rerun_202609/glycoQTL_tmem106b/glycoQTL_unadjusted_AD/colocboost/unadj_glyco_peptide_unrelated_AD.chr7_ENSG00000106460.cb_xqtl_AD_Bellenguez.rds")[[1]]
pos <- as.numeric(stringr::str_split(cb$data_info$variables, ":", simplify = TRUE)[, 2])
idx <- which(pos >= win[1] & pos <= win[2]); stopifnot(length(idx) > 0)
print(cb$data_info$outcome_info$outcome_names)   # y1 pQTL, y2-4 gp_8489-8491, y5 N2_score, y6 SAYVSYDVQK, y7 AD
edb <- EnsDb.Hsapiens.v86::EnsDb.Hsapiens.v86
tx <- ensembldb::transcripts(edb, filter = AnnotationFilter::GRangesFilter(GenomicRanges::GRanges("7", IRanges::IRanges(win[1], win[2]))), columns = c("tx_id", "gene_id", "gene_name"))
tx <- tx[S4Vectors::mcols(tx)$gene_name == "TMEM106B"]; tx <- tx[which.max(BiocGenerics::width(tx))]
ex_tx <- ensembldb::exonsBy(edb, by = "tx", filter = AnnotationFilter::TxIdFilter(tx$tx_id))[[1]]
ex_df <- as.data.frame(ex_tx)
intron_df <- data.frame(start = head(IRanges::end(ex_tx), -1), end = tail(IRanges::start(ex_tx), -1), y = 0)
dir.create("data", showWarnings = FALSE)
saveRDS(list(cb_res = cb, grange = range(idx), intron_df = intron_df, ex_df = ex_df), "data/Figure_4c_locuszoom.rds")

### d  unadjusted vs adjusted beta at rs3173615

In [ ]:
# Panel d: cis-glycoQTL effect of rs3173615
r <- readRDS("analysis_data/fig4d_cdomain_adjusted_qtl.rds")
q <- as.data.frame(r$q[r$q$adjustment %in% c("unadjusted", "adjusted"), c("gp", "adjustment", "beta", "se", "ci_lo", "ci_hi", "p", "n")])
stopifnot(nrow(q) == 6, all(q$n == 314), identical(r$variant, "chr7:12229791:C:G"))
saveRDS(list(q = q, variant = r$variant), "data/Figure_4d_qtl_adjusted.rds"); print(q)

### e S185/T185 ratio data (`data/Figure_4e_ratio.rds`)

In [ ]:
# Panel e: within-donor S185/T185 intensity ratio of N183 Man5-Man7 glycopeptides
suppressMessages({library(SummarizedExperiment)})
tmem <- readRDS("tmem106b/glyco/glyco_tmem106b.Rds")
het <- which(tmem$rs3173615 %in% "C/G"); ids <- as.character(tmem$ProjID[het]); stopifnot(length(ids) == 150)
features <- data.frame(glycan = paste0("HexNAc(2)Hex(", 5:7, ")"), T = c("TMEM_GP04", "TMEM_GP10", "TMEM_GP16"), S = c("TMEM_GP02", "TMEM_GP08", "TMEM_GP14"))
stopifnot(all(rowData(tmem)[features$T, "Peptide.Sequence"] == "LNNITIIGPLDMK"), all(rowData(tmem)[features$S, "Peptide.Sequence"] == "LNNISIIGPLDMK"),
          !any(grepl("15.9949", rowData(tmem)[c(features$T, features$S), "Modified.Sequence"])))   # unoxidized only
d <- do.call(rbind, lapply(seq_len(nrow(features)), function(i) {
  Tt <- assay(tmem, "intensity")[features$T[i], het]; Ss <- assay(tmem, "intensity")[features$S[i], het]
  stopifnot(all(is.finite(Tt)), all(is.finite(Ss)), all(Tt >= 0), all(Ss >= 0))
  paired <- Tt > 0 & Ss > 0; ratio <- rep(NA_real_, length(Tt)); ratio[paired] <- Ss[paired] / Tt[paired]   # ratio needs both alleles detected
  data.frame(ProjID = ids, glycan = features$glycan[i], T = Tt, S = Ss, ratio, log2_ratio = log2(ratio)) }))
d$glycan <- factor(d$glycan, levels = features$glycan)
wide <- reshape(d[c("ProjID", "glycan", "log2_ratio")], idvar = "ProjID", timevar = "glycan", direction = "wide")
tests <- do.call(rbind, lapply(combn(features$glycan, 2, simplify = FALSE), function(g) {
  delta <- wide[[paste0("log2_ratio.", g[1])]] - wide[[paste0("log2_ratio.", g[2])]]; delta <- delta[is.finite(delta)]
  data.frame(first = g[1], second = g[2], n = length(delta), mean_log2_difference = mean(delta), p = t.test(delta)$p.value) }))
tests$p_holm <- p.adjust(tests$p, "holm")
sumtab <- do.call(rbind, lapply(features$glycan, function(g) { z <- subset(d, glycan == g); tt <- t.test(z$log2_ratio, mu = 0)
  data.frame(glycan = g, n = sum(is.finite(z$ratio)), mean_log2_ratio = unname(tt$estimate), ci_lo = tt$conf.int[1], ci_hi = tt$conf.int[2], p_log2_vs_zero = tt$p.value) }))
sumtab$pct_lower <- 100 * (1 - 2^sumtab$mean_log2_ratio); sumtab$pct_lo <- 100 * (1 - 2^sumtab$ci_hi); sumtab$pct_hi <- 100 * (1 - 2^sumtab$ci_lo)
saveRDS(list(d = d, tests = tests, summary = sumtab), "data/Figure_4e_ratio.rds")
tests
sumtab 
mn <- function(x, d = 3) sub("-", "\u2212", sprintf(paste0("%.", d, "f"), x))
cat(sprintf("%s: S185 abundance was %.1f%% lower than T185 abundance (mean log\u2082 S185/T185 = %s; 95%% CI, %s to %s; n = %d; P = %s)",
            sub(".*Hex\\((\\d)\\)", "Man\\1", sumtab$glycan), sumtab$pct_lower, mn(sumtab$mean_log2_ratio), mn(sumtab$ci_lo), mn(sumtab$ci_hi), sumtab$n,
            format(sumtab$p_log2_vs_zero, digits = 4, scientific = TRUE)), sep = "\n")

#### Table 1: effect size with 95% CI and P value at the lead variant and at T185S (v202609 rerun)

In [ ]:
# Table 1 (v202609 rerun): glycopeptiforms of the panel c colocBoost object at the lead
library(colocboost)
cbd <- readRDS('data/Figure_4c_locuszoom.rds')$cb_res
cos_set <- cbd$cos_summary$colocalized_variables %>% str_split(., '; ') %>% unlist
message('#var in CoS: ', uniqueN(cos_set)); lead_var <- cos_set[1]; message('Lead variant in CoS: ', lead_var)
vars <- c(lead = lead_var, T185S = 'chr7:12229791:C:G')
var_idx <- setNames(match(vars, cbd$data_info$variables), names(vars)); stopifnot(!anyNA(var_idx))
gp_ids <- paste0('gp_', 8489:8491); gp_outcomes <- paste0('DLPFC_Klein_gpQTL_unadjusted_', gp_ids); stopifnot(all(gp_outcomes %in% names(cbd$data_info$z)))

ad_cos_out <- paste(cbd$cos_summary$colocalized_outcomes[grepl('AD_Bellenguez', cbd$cos_summary$colocalized_outcomes)], collapse = '; ')
cb_tab1 <- rbindlist(lapply(names(vars), function(k) data.table(
  variant = k, variant_ID = vars[[k]], molecular_trait_id = gp_ids,
  coef_colocboost = sapply(gp_outcomes, function(nm) cbd$data_info$coef[[nm]][var_idx[[k]]]),
  z_colocboost    = sapply(gp_outcomes, function(nm) cbd$data_info$z[[nm]][var_idx[[k]]]),
  ad_coloc = sapply(gp_outcomes, function(nm) grepl(nm, ad_cos_out, fixed = TRUE)))))
cb_tab1[, p_colocboost := 2 * pnorm(-abs(z_colocboost))]
cb_n <- setNames(as.numeric(cbd$data_info$outcome_info$sample_size), cbd$data_info$outcome_info$outcome_names)
cb_tab1[, cb_n := cb_n[paste0('DLPFC_Klein_gpQTL_unadjusted_', molecular_trait_id)]]; stopifnot(!anyNA(cb_tab1$cb_n))
cb_tab1[, cb_beta := colocboost:::get_beta(z_colocboost, cb_n)]
cb_tab1[, `:=`(cb_ci_lo = tanh(atanh(cb_beta) - qnorm(.975) / sqrt(cb_n - 3)), cb_ci_hi = tanh(atanh(cb_beta) + qnorm(.975) / sqrt(cb_n - 3)))]

TQ <- 'data_output/TMEM106B_rerun_202609/tensorqtl_unadjusted/unadj_glyco.TMEM106B.sorted_chr7.cis_qtl.pairs.tsv.gz'
COVF <- 'data_input/unadjust/covariate/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'
tq_id <- function(v) sub(':([^:]+):([^:]+)$', '_\\1_\\2', v)   # chr7:pos:ref:alt -> chr7:pos_ref_alt (TensorQTL ID format)
tq <- fread(cmd = sprintf("zcat %s | awk -F'\\t' 'NR==1||$3==\"gp_8489\"||$3==\"gp_8490\"||$3==\"gp_8491\"'", TQ))[variant_id %in% tq_id(vars)]
stopifnot(nrow(tq) == length(gp_ids) * length(vars), all(tq$n == 316))
n_cov <- as.integer(system(sprintf('zcat %s | wc -l', COVF), intern = TRUE)) - 1L; cat('covariates:', n_cov, '\n')
tq[, dof := n - 2L - n_cov]                                  # TensorQTL residual degrees of freedom
stopifnot(isTRUE(all.equal(2 * pt(-abs(tq$bhat / tq$sebhat), tq$dof), tq$pvalue, tolerance = 1e-2)))   # recompute P from bhat/sebhat to validate dof and columns
tq[, `:=`(variant_ID = sub('_([^_]+)_([^_]+)$', ':\\1:\\2', variant_id), ci_lo = bhat - qt(.975, dof) * sebhat, ci_hi = bhat + qt(.975, dof) * sebhat)]   # t-based 95% CI, consistent with the TensorQTL P
cb_tab1 <- merge(cb_tab1, tq[, .(molecular_trait_id, variant_ID, beta = bhat, se = sebhat, ci_lo, ci_hi, pvalue, dof)], by = c('molecular_trait_id', 'variant_ID'))

cb_tab1 <- merge(cb_tab1, anno_ref[gpID %in% gp_ids, .(molecular_trait_id = gpID, peptide_ID, GlycanClass, GlycanComposition, GlycoSite)], by = 'molecular_trait_id')
cb_tab1[, GlycoSite := sub('.*_', '', GlycoSite)]                # Q9NUM4_N183 -> N183
setorder(cb_tab1, variant, molecular_trait_id)
saveRDS(cb_tab1, 'data/Table1_colocboost_effect_6peptiforms.rds')
cb_tab1[, .(variant, variant_ID, molecular_trait_id, GlycanComposition, ad_coloc, coef_colocboost, cb_beta, z_colocboost, p_colocboost, beta, se, ci_lo, ci_hi, pvalue)]

### Genetic effects of the lead variant and T185S on TMEM106B glycopeptiforms

In [ ]:
# Table 1: TMEM106B glycopeptiforms
lv <- cb_tab1[variant == 'lead']; tv <- cb_tab1[variant == 'T185S']; stopifnot(identical(lv$molecular_trait_id, tv$molecular_trait_id))
fmt_ci <- function(b, lo, hi, d = 3) sprintf(paste0('%.', d, 'f [%.', d, 'f, %.', d, 'f]'), b, lo, hi)
ratio_tab <- as.data.table(readRDS('data/Figure_4e_ratio.rds')$summary)
ratio_tab[, ratio_ci := fmt_ci(2^mean_log2_ratio, 2^ci_lo, 2^ci_hi, 2)]
rt <- ratio_tab[match(lv$GlycanComposition, glycan)]; stopifnot(!anyNA(rt$n))
lead_lab <- paste0('Lead variant ', lv$variant_ID[1]); t_lab <- 'T185S rs3173615 (chr7:12229791:C:G)'
common <- list(`Glycan composition` = lv$GlycanComposition, `Glycosite` = lv$GlycoSite, `Glycan class` = lv$GlycanClass,
               `In AD colocalization signal` = ifelse(lv$ad_coloc, 'Yes', 'No'),
               `S185/T185 ratio in C/G donors, geometric mean [95% CI]` = rt$ratio_ci, `Donors with both forms (n)` = rt$n)
table1 <- as.data.table(c(common,
  setNames(list(fmt_ci(lv$beta, lv$ci_lo, lv$ci_hi), signif(lv$pvalue, 3), fmt_ci(tv$beta, tv$ci_lo, tv$ci_hi), signif(tv$pvalue, 3)),
           paste(c(lead_lab, lead_lab, t_lab, t_lab), c('TensorQTL beta [95% CI]', 'TensorQTL P value', 'TensorQTL beta [95% CI]', 'TensorQTL P value')))), check.names = FALSE)
fwrite(table1, 'data/Table1_TMEM106B_glycopeptiforms.csv')
table1_cb <- as.data.table(c(common,
  setNames(list(fmt_ci(lv$cb_beta, lv$cb_ci_lo, lv$cb_ci_hi), signif(lv$p_colocboost, 3), fmt_ci(tv$cb_beta, tv$cb_ci_lo, tv$cb_ci_hi), signif(tv$p_colocboost, 3)),
           paste(c(lead_lab, lead_lab, t_lab, t_lab), c('colocBoost standardized beta [95% CI]', 'colocBoost P value', 'colocBoost standardized beta [95% CI]', 'colocBoost P value')))), check.names = FALSE)
fwrite(table1_cb, 'data/Table1_TMEM106B_glycopeptiforms_colocboost.csv')
table1_cb

### f, g. Mediation of the T185S effect, forward and switched roles (checkpoint of `data/mediation.png` and `data/mediation_rev.png`)

In [ ]:
# Panels f and g: checkpoint of the numbers drawn in data/mediation.png
suppressMessages(library(mediation)); conflicts_prefer(base::unname)   # Matrix (attached by mediation) and S4Vectors also export unname
med_dir <- 'data_output/mediation'
sum_med <- function(file, panel) { m <- readRDS(file.path(med_dir, file)); s <- summary(m)
  data.frame(panel = panel, n = nobs(m$model.y), total = s$tau.coef, total_lo = s$tau.ci[1], total_hi = s$tau.ci[2], acme = s$d0, ade = s$z0, ade_lo = s$z0.ci[1], ade_hi = s$z0.ci[2],
             prop = s$n0, prop_lo = s$n0.ci[1], prop_hi = s$n0.ci[2], row.names = NULL) }
fg <- rbind(sum_med('TMEM106B_ctcnStrategy_12229791_SAYVSYDVQK_gp8490_v202609.rds', 'f'), sum_med('TMEM106B_ctcnStrategy_12229791_reverse_gp8490_v202609.rds', 'g'))
stopifnot(all(fg$n == 289), abs(fg$prop - c(0.29, 0.35)) < 0.005, abs(fg$total - c(-0.228, -0.401)) < 0.0005, abs(fg$ade + c(0.162, 0.261)) < 0.0005, abs(fg$ade_lo + c(0.230, 0.393)) < 0.0005, abs(fg$ade_hi + c(0.094, 0.132)) < 0.0005)
print(transform(fg, prop = round(prop, 3), prop_lo = round(prop_lo, 3), prop_hi = round(prop_hi, 3)), digits = 3)

## Supporting statistics for the TMEM106B results text

### S1. LD between the CoS lead (rs5011439) and the T185S coding variant (rs3173615)

In [ ]:
gbase <- 'FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.7'
gbim <- fread(paste0(gbase, '.bim'), header = FALSE)

library(BEDMatrix)
G <- BEDMatrix(gbase); colnames(G) <- gbim$V2
vv <- gbim[V4 %in% c(12229185, 12229791, 12232490, 12228456, 12229042), V2] # lead: 12229042, T185S:12229791
r2 <- cor(G[, vv], use = 'pairwise.complete.obs')^2
dimnames(r2) <- list(gbim[match(vv, V2), V4], gbim[match(vv, V2), V4])
round(r2, 3)


r2[c('12229042','12229791'), c('12229042','12229791')]
 

### S2. Peptide-level QTL effect by TMEM106B domain

In [ ]:
qres <- as.data.table(readRDS('data/Figure_4b_pqtl_effects.rds'))
pep_start <- c(EDAYDGVTSENMR = 15, NGDVSQFPYVEFTGR = 42, DSVTCPTCQGTGR = 57,
               GQENQLVALIPYSDQR = 73, SAYVSYDVQK = 130, YQYVDCGR = 248)
qres[, start := pep_start[peptide]]
qres[, domain := ifelse(start >= 118, 'C-terminal (luminal)', 'N-terminal (cytoplasmic)')]
setorder(qres, p_joint); qres[]
# (2) SAYVSYDVQK (residues 130-139, C-terminal luminal domain)